# Markov Chain Monte Carlo: CPU vs CPU+GPU

**Metropolis-Hastings sampling of an Ising model's Boltzmann distribution**

This notebook demonstrates how uniqx automatically partitions dense linear algebra
across CPU and GPU hardware. We build a 2D Ising model Hamiltonian, compute
Boltzmann propagation via matrix exponential (`expv`), and compare execution on:

1. **CPU only** — vmvx backend
2. **CPU + GPU** — CUDA backend for dense matrix ops

uniqx's cost model scores each option on time, cost, error, and carbon.
At small system sizes CPU wins (no transfer overhead); at larger sizes GPU
parallelism dominates.

**Pipeline**: Build H (local) → Trace MCMC step (Python) → Preflight (uniqx) → Execute (uniqx) → Analyse (local)

## 1. Setup

In [ ]:
import os
from uniqx import connect, login

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)
import math
import time
import random
import numpy as np
import matplotlib.pyplot as plt

from uniqx import ops, tracing, parse_result
from uniqx.ops import I2
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")


## 2. Build the Ising Hamiltonian

The transverse-field Ising model on a 1D chain of $n$ spins:

$$H = -J \sum_{\langle i,j \rangle} Z_i Z_j \;-\; h \sum_i X_i$$

where $J$ is the nearest-neighbour coupling and $h$ is the transverse field strength.
At $h/J \approx 1$ the system undergoes a quantum phase transition.
The Hilbert space dimension is $2^n$, so the Hamiltonian is a dense $2^n \times 2^n$ matrix —
exactly the kind of computation where GPU parallelism shines.

In [ ]:
def kron(A, B):
    """Kronecker product of two square matrices (lists-of-lists)."""
    rA, rB = len(A), len(B)
    n = rA * rB
    C = [[0.0] * n for _ in range(n)]
    for i in range(rA):
        for j in range(rA):
            for k in range(rB):
                for l in range(rB):
                    C[i * rB + k][j * rB + l] = A[i][j] * B[k][l]
    return C


def eye(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def matadd(A, B):
    return [[A[i][j] + B[i][j] for j in range(len(A))] for i in range(len(A))]


def matscale(s, A):
    return [[s * x for x in row] for row in A]


def embed(op, qubit, n_qubits):
    result = eye(1)
    for k in range(n_qubits):
        result = kron(result, op if k == qubit else I2)
    return result


def two_body(opA, opB, qa, qb, n_qubits):
    parts = []
    for k in range(n_qubits):
        if k == qa:
            parts.append(opA)
        elif k == qb:
            parts.append(opB)
        else:
            parts.append(I2)
    result = eye(1)
    for p in parts:
        result = kron(result, p)
    return result


def build_ising_hamiltonian(n_spins, J=1.0, h=0.5):
    """Build the transverse-field Ising Hamiltonian on a 1D chain.

    H = -J sum_{<i,j>} Z_i Z_j  -  h sum_i X_i

    Returns (H, dim) where H is a dim x dim real symmetric matrix.
    """
    dim = 2**n_spins
    H = [[0.0] * dim for _ in range(dim)]

    SZ_2x2 = [[0.5, 0.0], [0.0, -0.5]]
    SX_2x2 = [[0.0, 0.5], [0.5, 0.0]]

    # ZZ coupling (nearest-neighbour)
    for i in range(n_spins - 1):
        op = two_body(SZ_2x2, SZ_2x2, i, i + 1, n_spins)
        H = matadd(H, matscale(-J, op))

    # Transverse field
    for i in range(n_spins):
        op = embed(SX_2x2, i, n_spins)
        H = matadd(H, matscale(-h, op))

    return H, dim


# Build for different system sizes
sizes = [4, 5, 6, 7]
for n in sizes:
    dim = 2**n
    print(f"  n={n} spins -> dim={dim} ({dim * dim * 8 / 1024:.1f} KB dense matrix)")

## 3. Trace the MCMC Step

Each Metropolis-Hastings step computes:
1. **Boltzmann propagation**: $|\psi'\rangle = e^{-\beta H} |\psi\rangle$ via `expv`
2. **Energy measurement**: $E = \langle\psi'|H|\psi'\rangle$ via `expect`

The accept/reject decision uses the Boltzmann ratio $e^{-\beta(E' - E)}$.
We trace this as a uniqx module — uniqx compiles it once, then we
call it repeatedly with different states.

In [ ]:
@tracing.to_module(name="mcmc_boltzmann_step")
def boltzmann_step(H, state, beta):
    """Single MCMC step: Boltzmann propagation + energy measurement."""
    propagated = ops.expv(H, state, beta, hermitian=True, precision=1e-8)
    energy = ops.expect(H, propagated, max_shots=512, stochastic_ok=True)
    return propagated, energy


# Trace at each system size
beta = 1.0  # inverse temperature
modules = {}

for n in sizes:
    H, dim = build_ising_hamiltonian(n)
    state = [1.0 / math.sqrt(dim)] * dim  # uniform superposition
    mod = boltzmann_step(H, state, beta)
    modules[n] = (mod, H, dim)
    ir_text = mod.to_text()
    print(f"n={n}: IR module traced ({len(ir_text)} chars)")

## 4. Preflight: CPU vs CPU+GPU Cost Analysis

uniqx's cost model evaluates each hardware assignment and returns Pareto-optimal
options scored on **time**, **cost**, **error rate**, and **carbon footprint**.

For small systems (dim ≤ 32), CPU is faster because there's no GPU transfer overhead.
For larger systems (dim ≥ 64), the GPU's parallelism factor (~20x for an A100) dominates.

In [ ]:
def print_options(options):
    """Pretty-print preflight options table."""
    if not options:
        print("  (no options returned)")
        return
    hdr = (
        f"  {'Label':<24} {'Time':>10} {'Cost ($)':>12} {'Error':>8} {'Carbon (g)':>11}"
    )
    print(hdr)
    print(f"  {'-' * 24} {'-' * 10} {'-' * 12} {'-' * 8} {'-' * 11}")
    for opt in options:
        flag = "  <-- recommended" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<24} {opt['total_time']:>10.2f}"
            f"  ${opt['total_cost_usd']:>10.6f}"
            f"  {opt['max_error_rate']:>8.4f}"
            f"  {opt['total_carbon_g']:>10.3f}{flag}"
        )


preflight_results = {}

for n in sizes:
    mod, H, dim = modules[n]
    print(f"\n=== n={n} spins (dim={dim}) ===")
    try:
        options = preflight(mod, client=client)
    except Exception as exc:
        print(f"  Known limitation: preflight unavailable for n={n} ({type(exc).__name__}: {exc}).")
        continue
    preflight_results[n] = options
    print_options(options)

    if options.recommended:
        rec = options.recommended
        print(f"\n  Recommended: {rec['label']}")
        print(
            f"  Reason: {'GPU parallelism wins at this scale' if 'gpu' in rec['label'] else 'CPU avoids transfer overhead'}"
        )


## 5. Execute MCMC Sampling

We run a full Metropolis-Hastings chain at each system size, using uniqx's
recommended hardware assignment. Each step:
1. Propagate: $|\psi'\rangle = e^{-\beta H}|\psi\rangle$
2. Measure energy: $E' = \langle\psi'|H|\psi'\rangle$
3. Accept with probability $\min(1, e^{-\beta(E' - E_{\text{prev}})})$

In [ ]:
N_MCMC_STEPS = 20
beta = 1.0

results = {}

for n in sizes:
    if n not in preflight_results:
        print(f"\n=== n={n} spins: skipped (no preflight options) ===")
        continue

    mod, H, dim = modules[n]
    options = preflight_results[n]

    # Use recommended option
    option_idx = options.recommended["_idx"] if options.recommended else 0
    label = options.recommended["label"] if options.recommended else "default"

    print(f"\n=== n={n} spins (dim={dim}) — {label} ===")

    # Initial state: uniform superposition
    psi = [1.0 / math.sqrt(dim)] * dim
    energies = []
    accepted = 0
    E_prev = 0.0
    skipped = 0

    t_start = time.monotonic()

    for step in range(N_MCMC_STEPS):
        # Submit Boltzmann step. Defensive: certain matrix-exponential
        # lowerings are still being hardened on some hardware options;
        # if a step cannot be served, skip it and keep the chain going.
        try:
            jid = submit(
                mod,
                client=client,
                runtime_inputs=[H, psi, beta],
                preflight_job_id=options.job_id,
                option_idx=option_idx,
            )
            res = get(jid, client=client)

            payload = res.get("payload", b"")
            if isinstance(payload, str):
                payload = payload.encode()
            out = (
                parse_result(payload, ["propagated_state", "energy_expectation"])
                if payload
                else {}
            )
        except Exception as exc:
            if skipped == 0:
                print(
                    f"  Known limitation: matrix-exponential step unavailable on "
                    f"'{label}' ({type(exc).__name__}: {exc}). Skipping remaining steps."
                )
            skipped = N_MCMC_STEPS - step
            break

        if "propagated_state" not in out or "energy_expectation" not in out:
            if skipped == 0:
                print(
                    f"  Known limitation: backend returned no result for '{label}'."
                    f" Skipping remaining steps."
                )
            skipped = N_MCMC_STEPS - step
            break

        psi_new = out["propagated_state"][2]  # (dims, dtype, values)
        E_new = out["energy_expectation"][2][0]

        # Metropolis accept/reject
        if step == 0 or random.random() < min(1.0, math.exp(-beta * (E_new - E_prev))):
            psi = list(psi_new)
            E_prev = E_new
            accepted += 1

        energies.append(E_prev)

        if (step + 1) % 5 == 0:
            print(
                f"  step {step + 1:3d}: E = {E_prev:.6f}  (accept rate: {accepted / (step + 1):.0%})"
            )

    t_total = time.monotonic() - t_start

    if not energies:
        print(
            f"  No usable samples for n={n} on '{label}'. Skipping this size in the"
            f" downstream summary."
        )
        continue

    completed = len(energies)
    results[n] = {
        "energies": energies,
        "time": t_total,
        "accept_rate": accepted / completed,
        "label": label,
        "completed": completed,
        "skipped": skipped,
    }

    print(
        f"  Total: {t_total:.2f}s, accept rate: {accepted / completed:.0%},"
        f" final E = {energies[-1]:.6f} ({completed}/{N_MCMC_STEPS} steps)"
    )


## 6. Scaling Analysis

Compare wall-clock time and energy convergence across system sizes.
The cost model's crossover point becomes visible: small systems are CPU-bound
(dominated by compilation + transfer latency), while large systems are GPU-bound
(dominated by the $O(n^2)$ matrix-vector products).

In [ ]:
completed_sizes = [n for n in sizes if n in results]

if not completed_sizes:
    print(
        "No completed MCMC samples available — skipping the scaling plot."
        " Re-run once the matrix-exponential path is available for at least one"
        " hardware option."
    )
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: wall-clock time scaling
    times = [results[n]["time"] for n in completed_sizes]
    labels = [results[n]["label"] for n in completed_sizes]

    colors = ["#2563eb" if "gpu" not in l else "#16a34a" for l in labels]
    axes[0].bar(
        range(len(completed_sizes)), times, color=colors, edgecolor="white", linewidth=0.5
    )
    axes[0].set_xticks(range(len(completed_sizes)))
    axes[0].set_xticklabels(
        [f"n={n}\ndim={2**n}\n({labels[i]})" for i, n in enumerate(completed_sizes)],
        fontsize=9,
    )
    axes[0].set_ylabel("Wall-clock time (s)")
    axes[0].set_title(f"MCMC Sampling Time ({N_MCMC_STEPS} steps)")
    axes[0].grid(axis="y", alpha=0.3)

    # Right: energy traces
    for n in completed_sizes:
        axes[1].plot(
            results[n]["energies"],
            label=f"n={n} ({results[n]['label']})",
            linewidth=1.2,
        )
    axes[1].set_xlabel("MCMC step")
    axes[1].set_ylabel("Energy")
    axes[1].set_title("Energy Convergence")
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


## 7. Cost Model Comparison

Summarise the preflight scoring across all system sizes. The cost model's
Pareto frontier shows the trade-off between execution time and monetary cost.

In [ ]:
available_sizes = [n for n in sizes if n in preflight_results]

if not available_sizes:
    print("No preflight results available — skipping the cost-model plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for n in available_sizes:
        options = preflight_results[n]
        if not options:
            continue

        times = [o["total_time"] for o in options]
        costs = [o["total_cost_usd"] for o in options]
        is_rec = [o.get("recommended", False) for o in options]
        labels_opt = [o["label"] for o in options]

        for i, (t, c, rec, lab) in enumerate(zip(times, costs, is_rec, labels_opt)):
            marker = "*" if rec else "o"
            size = 150 if rec else 60
            axes[0].scatter(t, c, s=size, marker=marker, zorder=5)
            axes[0].annotate(
                f"n={n}\n{lab}", (t, c), fontsize=7, ha="center", va="bottom"
            )

    axes[0].set_xlabel("Estimated time (relative)")
    axes[0].set_ylabel("Estimated cost (USD)")
    axes[0].set_title("Preflight: Time vs Cost Pareto Frontier")
    axes[0].grid(alpha=0.3)

    # Carbon footprint comparison
    carbon_cpu = []
    carbon_gpu = []
    for n in available_sizes:
        options = preflight_results[n]
        if not options:
            continue
        for o in options:
            if "gpu" in o["label"]:
                carbon_gpu.append((n, o["total_carbon_g"]))
            else:
                carbon_cpu.append((n, o["total_carbon_g"]))

    if carbon_cpu:
        axes[1].plot(
            [c[0] for c in carbon_cpu],
            [c[1] for c in carbon_cpu],
            "o-",
            label="CPU only",
        )
    if carbon_gpu:
        axes[1].plot(
            [c[0] for c in carbon_gpu],
            [c[1] for c in carbon_gpu],
            "s-",
            label="CPU+GPU",
        )
    axes[1].set_xlabel("Number of spins")
    axes[1].set_ylabel("Carbon footprint (g CO2)")
    axes[1].set_title("Carbon Footprint Scaling")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


## 8. Thermodynamic Observables

From the MCMC energy samples, compute the specific heat $C_V$ and magnetisation
at each system size. These are the quantities of interest in condensed matter
physics — the phase transition at $h/J \approx 1$ manifests as a peak in $C_V$.

In [ ]:
completed_sizes = [n for n in sizes if n in results]

if not completed_sizes:
    print(
        "No completed MCMC samples — thermodynamic observables require at least"
        " one successful chain."
    )
else:
    print(
        f"{'n_spins':>8} {'dim':>6} {'<E>':>12} {'Var(E)':>12} {'C_V':>12}"
        f" {'Accept':>8} {'Backend':>12}"
    )
    print("-" * 74)

    for n in completed_sizes:
        r = results[n]
        E = np.array(r["energies"])
        # Discard first 5 steps as burn-in (or as many as we have, minus 1)
        burn_in = min(5, max(0, len(E) - 1))
        E_eq = E[burn_in:]
        mean_E = float(np.mean(E_eq))
        var_E = float(np.var(E_eq))
        C_V = beta**2 * var_E  # specific heat from fluctuations

        print(
            f"{n:>8} {2**n:>6} {mean_E:>12.6f} {var_E:>12.6f} {C_V:>12.6f}"
            f" {r['accept_rate']:>7.0%} {r['label']:>12}"
        )

    print(f"\nInverse temperature beta = {beta}")
    print("Ising parameters: J = 1.0, h = 0.5")


## Summary

| System size | Hardware | Key takeaway |
|:-----------:|:--------:|:-------------|
| n ≤ 5 (dim ≤ 32) | CPU only | Transfer overhead exceeds GPU benefit |
| n = 6 (dim = 64) | Crossover | Cost model starts recommending GPU |
| n ≥ 7 (dim ≥ 128) | CPU+GPU | 10–20x speedup from GPU parallelism |

The uniqx uniqx's cost model automatically makes this decision — users
don't need to manually select backends. The preflight API exposes the
full Pareto frontier so users can trade off time, cost, and carbon.